For each subject and each MEG session/run in MEG_manifest.csv, this script:
- Locates that session’s volumetric source estimate file (your VolSourceEstimate, saved as *-stc.h5) under: '.../{main}/{SOURCE_ESTIMATES_DIR}/<subject_ID>/...'
- Locates the matching session’s source space file (RAS frame, *-source_space_RAS.fif) under: '.../{main}/{SOURCE_SPACES_DIR}/<subject_ID>/...'
- Computes a morph from the subject’s source space to a target template (e.g., fsaverage), and applies it to the STC.
- Writes a morphed NIfTI volume per session: '.../{main}/{MORPH_OUTPUT_DIR}/<subject_ID>/<subject_ID>_<session_ID>-aligned.nii'

[Runtime: approx. 10-12 min per MEG scan file]

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, re, mne, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mne
import nibabel as nib
from mne.io.constants import FIFF
from nilearn.datasets import load_mni152_template
from nilearn import plotting, image
import warnings
import csv

In [ ]:
##### SET UP ENVIRONMENTAL VARIABLES FOR FREESURFER:
freesurfer_config = config.get("freesurfer", {})
FREESURFER_HOME = Path(freesurfer_config.get("home", "/opt/freesurfer-7.4.1")).expanduser()
FS_LICENSE = Path(freesurfer_config.get("license", FREESURFER_HOME / "license.txt")).expanduser()
SUBJECTS_DIR = Path(freesurfer_config.get("subjects_dir", FREESURFER_HOME / "subjects")).expanduser()
CHECK_FS_VERSION = bool(freesurfer_config.get("check_version", True))
if not FREESURFER_HOME.exists():
    raise FileNotFoundError(f"FREESURFER_HOME not found: {FREESURFER_HOME}")
if not FS_LICENSE.exists():
    raise FileNotFoundError(f"FreeSurfer license file not found: {FS_LICENSE}")
if not SUBJECTS_DIR.exists():
    raise FileNotFoundError(f"FreeSurfer SUBJECTS_DIR not found: {SUBJECTS_DIR}")
os.environ["FREESURFER_HOME"] = str(FREESURFER_HOME)
os.environ["FS_LICENSE"] = str(FS_LICENSE)
os.environ["SUBJECTS_DIR"] = str(SUBJECTS_DIR)
fs_bin_dir = FREESURFER_HOME / "bin"
os.environ["PATH"] = f"{fs_bin_dir}:{os.environ.get('PATH', '')}"

subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
# --------------------------------------------------------------------
### SET PARAMETERS:

HARD_STOP = config['hard_errors']
RANDOM_SEED = config['random_seed']

SUBSET = config['subset']

OVERWRITE_ALIGNMENT = config['overwrite_alignment']

##### Disable MNE's filename-related warnings:
warnings.filterwarnings(
    "ignore",
    message="This filename .* does not conform to MNE naming conventions.*",
    category=RuntimeWarning)


### ALIGNMENT PARAMETERS:
SPACING = config['alignment_parameters']['spacing']
TARGET_TEMPLATE = config['alignment_parameters']['target_template']


# --------------------------------------------------------------------
### SET PATHS:

ROOT_DIR = Path(config['root_output_directory'])

### INPUTS:
RUN_MANIFEST_PATH = Path(ROOT_DIR) / 'subject_manifest.csv'
MEG_DATA_DIR = config['MEG_data_directory']
MEG_PARAMETERS_PATH = Path(ROOT_DIR) / 'MEG_manifest.csv'

COREGISTRATION_DATA_DIRECTORY = Path(ROOT_DIR) / config['coreg_output_dir']

SOURCE_SPACES_DIR = ROOT_DIR / config['source_space_output_dir']

SOURCE_ESTIMATES_DIR = ROOT_DIR / config['source_estimate_output_dir']

### OUTPUTS:
MORPH_OUTPUT_DIR = Path(ROOT_DIR) / config['morph_output_dir']
os.makedirs(MORPH_OUTPUT_DIR, exist_ok=True)


# --------------------------------------------------------------------
### INITIALIZATION:

# Load DataFrames:
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
MEG_runs = pd.read_csv(MEG_PARAMETERS_PATH)

In [ ]:
### DIAGNOSTIC SUBSETTING (if enabled):
if type(SUBSET) == int and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    MEG_runs = MEG_runs.head(SUBSET).copy()
    display(MEG_runs)

---------

In [ ]:
# =========================
# Morph volumetric Source Estimates to fsaverage
# =========================

# Discover unique subjects represented in MEG_runs.
# NOTE: .unique() preserves order of first appearance in the column.
subject_IDs = (
    MEG_runs["subject_ID"]
    .dropna()
    .astype(str)
    .unique())

print(
    "\n --> Processing MEG_runs entries for "
    f"{len(subject_IDs)} unique subjects ({len(MEG_runs)} total MEG runs)...")

for target_subject_ID in subject_IDs:

    # All MEG runs for this subject (one row per <subject_ID, MEG_session_ID, ...>).
    subject_runs = MEG_runs[MEG_runs["subject_ID"] == target_subject_ID].copy()

    if subject_runs.empty:
        # Should not really happen, but keep it safe.
        print(f"\n[warn] No MEG_runs entries found for subject_ID '{target_subject_ID}'. Skipping.")
        continue

    # Sort sessions in a deterministic way (chronological / lexical).
    if "MEG_session_ID" in subject_runs.columns:
        subject_runs = subject_runs.sort_values("MEG_session_ID")

    # Set paths to subject-specific directories.
    freesurfer_subject_directory = SUBJECTS_DIR / target_subject_ID
    subject_source_space_directory = SOURCE_SPACES_DIR / target_subject_ID
    subject_source_estimate_directory = SOURCE_ESTIMATES_DIR / target_subject_ID

    if not subject_source_estimate_directory.exists():
        message = (
            f"\n!!! No source estimate directory found for subject {target_subject_ID}:\n"
            f"    {subject_source_estimate_directory}")
        if HARD_STOP:
            raise FileNotFoundError(message)
        else:
            print(message)
            continue

    if not subject_source_space_directory.exists():
        message = (
            f"\n!!! No source space directory found for subject {target_subject_ID}:\n"
            f"    {subject_source_space_directory}")
        if HARD_STOP:
            raise FileNotFoundError(message)
        else:
            print(message)
            continue

    # ------------------------------------------------------------------
    # 1) Gather Source Estimate (.h5) and Source Space (.fif) per session
    # ------------------------------------------------------------------

    subject_stc_files = {}   # { session_ID: Path to STC file }
    subject_src_files = {}   # { session_ID: Path to SRC file }

    for _, run_row in subject_runs.iterrows():
        # Use MEG_session_ID column from MEG_runs as the session identifier:
        session_ID = str(run_row["MEG_session_ID"])

        # ---- Source Estimate target-file-discovery ----
        # We specifically require the *de-vectorized, time-resolved 4D* output from the previous stage.
        #    --> Naming convention (hard-coded upstream): "<subject_ID>-<session_ID>-...4D...-stc.h5"
        expected_stc_prefix = f"{target_subject_ID}-{session_ID}"

        candidate_stc_files = [
            filename
            for filename in os.listdir(subject_source_estimate_directory)
            if (
                filename.startswith(expected_stc_prefix)
                and ("4D" in filename)              # <-- critical disambiguator
                and filename.endswith("-stc.h5"))]

        if len(candidate_stc_files) == 0:
            message = (
                f"\n!!! No *4D* Source Estimate file found for subject {target_subject_ID}, "
                f"session '{session_ID}' in:\n"
                f"    {subject_source_estimate_directory}\n"
                f"    (Expected: prefix '{expected_stc_prefix}', contains '4D', endswith '-stc.h5')")
            if HARD_STOP:
                raise FileNotFoundError(message)
            else:
                print(message)
                continue

        if len(candidate_stc_files) > 1:
            print(
                f"\n[warn] Multiple *4D* Source Estimate files found for "
                f"{target_subject_ID}, session '{session_ID}'. "
                f"Using the first (sorted) match.")
            print("       Matches:")
            for fn in sorted(candidate_stc_files):
                print("       -", fn)

        stc_filename = sorted(candidate_stc_files)[0]
        subject_stc_files[session_ID] = subject_source_estimate_directory / stc_filename

        # ---- Source Space (.fif) discovery ----
        # Old naming: "<subject_ID>-<session_ID>-source_space_RAS.fif"
        expected_src_filename = f"{target_subject_ID}-{session_ID}-source_space_RAS.fif"
        expected_src_filepath = subject_source_space_directory / expected_src_filename

        if expected_src_filepath.exists():
            subject_src_files[session_ID] = expected_src_filepath
        else:
            message = (
                f"Expected Source Space file not found for subject {target_subject_ID}, "
                f"session '{session_ID}':\n    {expected_src_filepath}")
            # Old behavior: immediate hard error for missing src.
            raise FileNotFoundError(message)

    # Sort dictionaries by session_ID so processing order is deterministic.
    subject_stc_files = dict(sorted(subject_stc_files.items()))
    subject_src_files = dict(sorted(subject_src_files.items()))

    print(
        "__________________________________________________________________________________________________",
        "\n",
        f"Found the following {len(subject_stc_files.items())} MEG sessions "
        f"for subject_ID {target_subject_ID}:")
    for session_ID in subject_stc_files.keys():
        print("   -", session_ID)
    print()

    # Sanity check: ensure every STC session has a matching SRC.
    if set(subject_stc_files.keys()) != set(subject_src_files.keys()):
        missing_src_sessions = set(subject_stc_files.keys()) - set(subject_src_files.keys())
        message = (
            f"\n!!! Mismatch between STC and SRC sessions for subject {target_subject_ID}.\n"
            f"    Sessions with STC but no SRC: {sorted(missing_src_sessions)}")
        raise RuntimeError(message)

    print(
        f"\t --> Sanity Check passed: All {len(subject_src_files)} Source Estimate files for "
        f"subject {target_subject_ID} have corresponding Source Space files.\n")

    # ------------------------------------------------------------------
    # 2) Actual file-processing (morphing) starts here
    # ------------------------------------------------------------------

    for session_ID in subject_stc_files.keys():

        try:
            print(
                "___________________________________________________________________________________________________\n",
                f"Setting filepaths for session: '{target_subject_ID}_{session_ID}':")
            print("\tFreeSurfer subject directory:\t\t", freesurfer_subject_directory)
            print("\tSource Space directory:\t\t\t", subject_source_space_directory)
            print("\tSource Estimate file:\t\t\t", subject_stc_files[session_ID])
            print("\tSource Space (RAS) file:\t\t", subject_src_files[session_ID])
            print()
        except Exception as e:
            print(
                "\n\n!!! ERROR: cannot locate necessary data objects for subject; "
                "check file-pathing.\n\tError message:", e, "\n\n")
            if HARD_STOP:
                raise
            else:
                continue

        # Set path for morphing output.
        morph_output_directory = MORPH_OUTPUT_DIR / target_subject_ID
        os.makedirs(morph_output_directory, exist_ok=True)

        output_filepath = morph_output_directory / f"{target_subject_ID}_{session_ID}-aligned.nii"

        # Skip file if it already exists AND 'OVERWRITE_ALIGNMENT' is False.
        if output_filepath.exists() and not OVERWRITE_ALIGNMENT:
            print(
                f"\n*** !!!: Skipping {session_ID} "
                f"(morph output already exists and OVERWRITE_ALIGNMENT=False)\n")
            continue

        print(f"Starting morphing procedure for subject '{target_subject_ID}-{session_ID}'...")

        # ---- Load current MEG session Source Space ----
        print("\n  --> Loading Source Space (RAS)...\n")
        source_space_path = subject_src_files[session_ID]
        source_space = mne.read_source_spaces(str(source_space_path))

        # ---- Load current MEG session Source Estimate ----
        print("\n  --> Loading Source Estimate...\n")
        source_estimate_path = subject_stc_files[session_ID]
        source_estimate = mne.read_source_estimate(str(source_estimate_path))

        # Sanity checks for 'source_estimate' object:
        print(f"\tSTC type: {type(source_estimate)}")
        print(f"\tData shape: {source_estimate.data.shape}  # (n_voxels, n_times)")
        print(f"\tNumber of voxels: {source_estimate.data.shape[0]}")
        print(f"\tNumber of time points: {source_estimate.data.shape[1]}")
        assert isinstance(source_estimate, mne.VolSourceEstimate), "Source estimate is not volumetric!"

        # --- Sanity check: confirm that STC is in MRI coordinates ---
        print("\tPerforming sanity check on coordinate frame of Source Estimate...")
        if hasattr(source_estimate, "coord_frame"):
            frame = source_estimate.coord_frame
        else:
            # Fallback method using the source space.
            frame = source_space[0]["coord_frame"]

        if frame != FIFF.FIFFV_COORD_MRI:
            raise RuntimeError(
                "\n\n\n!!!: Source Estimate is *NOT* in MRI (surface RAS) coordinates; aborting.")
        print("\t   --> Source Estimate is in MRI (surface RAS) coordinates.\n")

        # Strict STC<-->SRC consistency check (for volumetric sources):
        try:
            src_verts = np.asarray(source_space[0]["vertno"], dtype=np.int64)
        except Exception as e:
            raise RuntimeError(
                f"Could not read source_space[0]['vertno'] for {target_subject_ID}-{session_ID}: {e}")

        stc_verts_raw = source_estimate.vertices

        # VolSourceEstimate.vertices is typically a 1-element list: [vertno]
        if isinstance(stc_verts_raw, (list, tuple)):
            if len(stc_verts_raw) != 1:
                raise RuntimeError(
                    f"Unexpected VolSourceEstimate.vertices length for {target_subject_ID}-{session_ID}: "
                    f"len(vertices)={len(stc_verts_raw)} (expected 1).")
            stc_verts = np.asarray(stc_verts_raw[0], dtype=np.int64)
        else:
            # Fallback: already array-like
            stc_verts = np.asarray(stc_verts_raw, dtype=np.int64)

        if stc_verts.size == 0:
            raise RuntimeError(
                f"STC vertices are empty for {target_subject_ID}-{session_ID}. "
                "This suggests a corrupted or invalid VolSourceEstimate.")

        if (stc_verts.shape != src_verts.shape) or (not np.array_equal(stc_verts, src_verts)):
            # Helpful debugging: show overlap and a few examples
            stc_set = set(stc_verts.tolist())
            src_set = set(src_verts.tolist())
            inter = len(stc_set & src_set)
            only_stc = len(stc_set - src_set)
            only_src = len(src_set - stc_set)

            raise RuntimeError(
                f"STC<->SRC vertex mismatch for {target_subject_ID}-{session_ID}.\n"
                f"  STC n_verts={stc_verts.size} | SRC n_verts={src_verts.size}\n"
                f"  overlap={inter} | only_in_stc={only_stc} | only_in_src={only_src}\n"
                "This indicates the STC file does not correspond to the SRC file "
                "(e.g., regenerated source space, wrong session pairing, or mixed pipelines).")

        print(f"\t   --> STC<->SRC check passed: n_verts={stc_verts.size}\n")

        # ---- Compute the morph ----
        print("Computing source morph to fsaverage...")
        morph = mne.compute_source_morph(
            src=source_space,
            subject_from=target_subject_ID,
            subject_to=TARGET_TEMPLATE,
            subjects_dir=SUBJECTS_DIR,
            spacing=SPACING,
            verbose=True)

        # ---- Apply morph and save as NIfTI ----
        print("Applying morph to Source Estimate and converting to NIfTI image...")
        img = morph.apply(source_estimate, output="nifti1")

        # Morph output sanity-check & report:
        try:
            out_dat = img.get_fdata(dtype=np.float32)
            finite_frac = float(np.isfinite(out_dat).mean()) if out_dat.size else 0.0
            max_abs = float(np.nanmax(np.abs(out_dat))) if out_dat.size else 0.0
            print(f"[MORPH OUT] nifti shape={out_dat.shape} | finite_frac={finite_frac:.6f} | maxabs={max_abs:.6g}")
        except Exception as e:
            print(f"[WARN] Could not compute morph output sanity stats for {target_subject_ID}-{session_ID}: {e}")

        print(f"Saving morphed Source Estimate as NIfTI to: {output_filepath}")
        img.to_filename(str(output_filepath))

        print(f"[SUCCESS] Morphing for {target_subject_ID}-{session_ID} complete.")